# 06 — Held-Out Evaluation

Evaluate the trained pipeline on **completely unseen data** from both datasets.

**Pipeline:**
1. Load best encoder + strategy prototypes (from notebook 04 `fewshot_{layer}.pt`)
2. Train shot-type classifier on **fine-tuned encoder** embeddings (SS train split)
3. **Part A — ShuttleSet held-out:** predict shot type on 3 unseen matches vs CSV ground truth
4. **Part B — FineBadminton held-out:** predict strategy on 10 unseen rallies vs labeled ground truth, plus shot-type accuracy
5. **Part C — Combined summary + CSV export for expert review**

**Two prediction tasks:**
- **Strategy** (5 classes) — ground truth only in FineBadminton
- **Shot type** (unified types) — ground truth in both FB (hit_type) and SS (CSV annotations)

Outputs:
- `heldout_ss_predictions_{layer}.csv` — per-shot predictions for SS held-out
- `heldout_fb_predictions_{layer}.csv` — per-shot predictions for FB held-out
- `heldout_expert_review_{layer}.csv` — combined, sorted by confidence (lowest first)
- Confusion matrices and distribution plots

In [ ]:
import os, sys, zipfile
from pathlib import Path

# ── Colab detection ───────────────────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_ROOT   = Path('/content/drive/MyDrive/Baddiev2')
    PROJECT_PATH = Path('/content/Baddiev2')
    ZIP_PATH     = DRIVE_ROOT / 'baddiev2_colab.zip'

    if not (PROJECT_PATH / 'src').exists():
        print('Extracting project files...')
        with zipfile.ZipFile(ZIP_PATH, 'r') as z:
            z.extractall(PROJECT_PATH)
        print(f'Extracted to {PROJECT_PATH}')
    else:
        print('Project already extracted.')

    sys.path.insert(0, str(PROJECT_PATH))
    os.chdir(PROJECT_PATH)

    import src.config as _cfg
    _cfg.MODELS_DIR            = DRIVE_ROOT / 'models'
    _cfg.RESULTS_DIR           = DRIVE_ROOT / 'results'
    _cfg.SS_SKELETONS_GDINO    = DRIVE_ROOT / 'datasets_preprocessing' / 'shuttleset_skeletons_gdino'
    _cfg.FB_SKELETONS_GDINO_V2 = DRIVE_ROOT / 'datasets_preprocessing' / 'finebadminton_skeletons_gdino_v2'
    _cfg.FB_ANNOTATIONS        = (
        DRIVE_ROOT / 'Datasets' / 'FineBadminton-dataset' / 'dataset' /
        'transformed_combined_rounds_output_en_evals_translated.json'
    )
    _cfg.SS_CSV_ROOT  = DRIVE_ROOT / 'datasets' / 'ShuttleSet' / 'set'
    _cfg.SS_MATCH_CSV = _cfg.SS_CSV_ROOT / 'match.csv'
    _cfg.SS_SPLIT_JSON = PROJECT_PATH / 'datasets_preprocessing' / 'shuttleset_split.json'

    _cfg.MODELS_DIR.mkdir(parents=True, exist_ok=True)
    _cfg.RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    print(f'Drive root   : {DRIVE_ROOT}')
    print(f'Models dir   : {_cfg.MODELS_DIR}')
    print(f'SS skeletons : {_cfg.SS_SKELETONS_GDINO}')
    print(f'FB skeletons : {_cfg.FB_SKELETONS_GDINO_V2}')
else:
    sys.path.insert(0, os.path.abspath('..'))
    DRIVE_ROOT = Path('..')
    print('Local run — using paths from config.py')

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json, csv
from tqdm import tqdm
from collections import Counter

from src.config import *
from src.data.graph_builder import GraphBuilder
from src.data.dataset import (
    FineBadmintonDataset, ShuttleSetDataset,
)
from src.data.feature_eng import FeatureEngineer
from src.models.stgcn_model import STGCN
from src.models.proto_net import PrototypicalNetwork

device = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps'  if torch.backends.mps.is_available() else
    'cpu'
)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Device: {device}')

## 1. Configuration

In [ ]:
# ── Ablation toggle ────────────────────────────────────────────────────────
FEATURE_LAYER = 'L2'     # Must match the layer used for SSL + fewshot training

# ── Held-out ShuttleSet matches (never seen during SSL or fine-tuning) ─────
HELD_OUT_SS_MATCHES = [
    "Carolina_Marin_An_Se_Young_TOYOTA_THAILAND_OPEN_2021_SemiFinals",
    "Anthony_Sinisuka_Ginting_Lee_Zii_Jia_HSBC_BWF_WORLD_TOUR_FINALS_2020_QuarterFinals",
    "Kento_MOMOTA_CHOU_Tien_Chen_Denmark_Open_2018_Finals",
]

# ── FB held-out split params (must match notebook 04) ──────────────────────
N_TRAIN_RALLIES = 30   # 30 train / 10 held-out (of 40 total)
FB_SPLIT_SEED   = 42

cfg = get_config('fewshot')
feature_dim = FEATURE_DIMS[FEATURE_LAYER]
cfg.stgcn.in_channels = feature_dim

print(f"Feature layer : {FEATURE_LAYER} ({feature_dim}-dim)")
print(f"Shot window   : {cfg.data.shot_window}")
print(f"Embedding dim : {cfg.stgcn.embedding_dim}")
print(f"\nHeld-out SS matches: {len(HELD_OUT_SS_MATCHES)}")
for m in HELD_OUT_SS_MATCHES:
    print(f"  {m}")

## 2. Load Encoder + Strategy Prototypes

In [ ]:
# Build graph and encoder
graph_builder = GraphBuilder(
    use_inter_player=cfg.ablation.use_inter_player,
    single_player=cfg.ablation.single_player,
)
adjacency = graph_builder.build_adjacency().to(device)

encoder = STGCN(
    in_channels=feature_dim,
    num_nodes=cfg.stgcn.num_nodes,
    adjacency=adjacency,
    num_layers=cfg.stgcn.num_layers,
    base_channels=cfg.stgcn.base_channels,
    embedding_dim=cfg.stgcn.embedding_dim,
    temporal_kernel=cfg.stgcn.temporal_kernel,
    dropout=cfg.stgcn.dropout,
).to(device)

# Load fewshot checkpoint (encoder weights + prototypes)
fewshot_path = MODELS_DIR / f'fewshot_{FEATURE_LAYER}.pt'
assert fewshot_path.exists(), f"Fewshot checkpoint not found: {fewshot_path}"

ckpt = torch.load(fewshot_path, map_location=device, weights_only=False)
encoder.load_state_dict(ckpt['encoder_state_dict'])
encoder.eval()

# Extract prototypes: dict {class_idx: tensor(256)} → stacked (NUM_CLASSES, 256)
proto_dict = ckpt['prototypes']
prototypes = torch.stack([proto_dict[i].to(device) for i in range(NUM_CLASSES)])
print(f"Prototypes: {prototypes.shape}  ({NUM_CLASSES} classes: {STRATEGY_CLASSES})")

proto_net = PrototypicalNetwork(encoder, distance=cfg.proto.distance)
print(f"Loaded: {fewshot_path.name}")

## 3. Train Shot-Type Classifier on Fine-Tuned Encoder

The shot-type classifier must be trained on the **fine-tuned** encoder's embeddings
(not the SSL-stage encoder, which has a different feature space).

We use the ShuttleSet train split (which has shot-type labels from CSVs) to fit
a LogisticRegression on the frozen fine-tuned encoder's embeddings.

In [ ]:
from sklearn.linear_model import LogisticRegression
import joblib

# Load SS train split to train shot-type classifier on fine-tuned embeddings
ss_train = ShuttleSetDataset(
    skeleton_dir=SS_SKELETONS_GDINO,
    shot_window=cfg.data.shot_window,
    feature_layer=FEATURE_LAYER,
    load_shot_types=True,
    split='train',
    split_json=SS_SPLIT_JSON,
)
print(f"SS train dataset: {len(ss_train)} samples")

# Extract embeddings using the fine-tuned encoder
encoder.eval()
train_embs, train_labels = [], []
with torch.no_grad():
    for i in tqdm(range(len(ss_train)), desc='Extracting SS train embeddings'):
        x, y = ss_train[i]
        if y < 0 or x.abs().sum() == 0:
            continue
        emb = encoder(x.unsqueeze(0).to(device)).cpu().numpy()[0]
        train_embs.append(emb)
        train_labels.append(y)

train_embs = np.array(train_embs)
train_labels = np.array(train_labels)
print(f"Valid labeled samples: {len(train_embs)} ({len(set(train_labels))} shot-type classes)")

# Train LogisticRegression on fine-tuned embeddings
shot_clf = None
if len(train_embs) > 0:
    shot_clf = LogisticRegression(max_iter=1000, C=1.0)
    shot_clf.fit(train_embs, train_labels)
    train_acc = shot_clf.score(train_embs, train_labels)
    print(f"\nShot-type classifier trained on fine-tuned encoder embeddings")
    print(f"  Train accuracy: {train_acc:.3f}")
    print(f"  Classes ({len(shot_clf.classes_)}): {[UNIFIED_SHOT_TYPES[i] for i in shot_clf.classes_]}")

    # Save for reuse
    clf_path = MODELS_DIR / f'shot_type_clf_finetuned_{FEATURE_LAYER}.joblib'
    joblib.dump(shot_clf, clf_path)
    print(f"  Saved: {clf_path.name}")
else:
    print("[WARN] No labeled SS train data — shot-type predictions will be skipped")
    print("  Check that shuttleset_split.json has matches in 'train' and skeletons exist")

## 4. Inference Helpers

In [ ]:
def predict_one(x_tensor, encoder, prototypes, shot_clf=None):
    """
    Run strategy + shot-type prediction on a single sample.

    Args:
        x_tensor: (C, T, V) feature tensor
        encoder: trained STGCN encoder (eval mode)
        prototypes: (NUM_CLASSES, D) strategy prototypes
        shot_clf: optional sklearn classifier for shot types

    Returns:
        dict with prediction results
    """
    with torch.no_grad():
        x = x_tensor.unsqueeze(0).to(device)  # (1, C, T, V)
        emb = encoder(x)  # (1, D)

        # Strategy prediction via ProtoNet distances
        distances = proto_net.compute_distances(emb, prototypes)  # (1, NUM_CLASSES)
        probs = F.softmax(-distances, dim=1)[0]  # (NUM_CLASSES,)
        sorted_probs, sorted_idx = probs.sort(descending=True)

        result = {
            'pred_strategy':     IDX_TO_STRATEGY[sorted_idx[0].item()],
            'strategy_conf':     sorted_probs[0].item(),
            'strategy_margin':   (sorted_probs[0] - sorted_probs[1]).item(),
            'strategy_2nd':      IDX_TO_STRATEGY[sorted_idx[1].item()],
            'strategy_2nd_conf': sorted_probs[1].item(),
        }

        # Shot-type prediction
        if shot_clf is not None:
            emb_np = emb.cpu().numpy()
            shot_probs = shot_clf.predict_proba(emb_np)[0]
            shot_pred_class = shot_clf.classes_[shot_probs.argmax()]
            result['pred_shot_type']  = UNIFIED_SHOT_TYPES[shot_pred_class]
            result['shot_type_conf']  = shot_probs.max()
        else:
            result['pred_shot_type']  = None
            result['shot_type_conf']  = None

    return result

# Quick sanity check
print("Inference helper ready.")

---
## Part A — ShuttleSet Held-Out Evaluation

Predict strategy + shot type on 3 completely unseen SS matches.
- **Shot type**: compare predictions vs CSV ground truth (measurable accuracy)
- **Strategy**: no SS ground truth — analyse predicted distributions and confidence only

In [ ]:
# Check skeleton and CSV availability for held-out matches
print(f"{'Match':<60} {'Skel?':>5} {'CSV?':>4} {'Frames':>7}")
print("-" * 80)

available_matches = []
for match in HELD_OUT_SS_MATCHES:
    skel_path = SS_SKELETONS_GDINO / match / 'skeletons.npy'
    fn_path   = SS_SKELETONS_GDINO / match / 'frame_nums.npy'
    csv_dir   = SS_CSV_ROOT / match
    has_skel  = skel_path.exists() and fn_path.exists()
    has_csv   = csv_dir.exists()

    n_frames = 0
    if has_skel:
        n_frames = len(np.load(fn_path))

    skel_icon = "YES" if has_skel else "NO"
    csv_icon  = "YES" if has_csv  else "NO"
    print(f"  {match[:58]:<58} {skel_icon:>5} {csv_icon:>4} {n_frames:>7}")

    if has_skel and has_csv:
        available_matches.append(match)
    elif not has_skel:
        print(f"    [SKIP] No GDINO skeletons — extract on Colab first")

print(f"\nAvailable for evaluation: {len(available_matches)}/{len(HELD_OUT_SS_MATCHES)} matches")

In [ ]:
# Load and run inference on held-out ShuttleSet matches
# Uses the same loading logic as ShuttleSetDataset whole_match mode
# Player ordering: P0=top court (nodes 0-16), P1=bottom court (nodes 17-33) — no swapping

from src.data.dataset import ShuttleSetDataset

shot_window = cfg.data.shot_window
feature_eng = FeatureEngineer(feature_layer=FEATURE_LAYER, homography=None)

ss_results = []

for match_name in tqdm(available_matches, desc='Matches'):
    # Load whole-match skeletons
    sk = np.load(str(SS_SKELETONS_GDINO / match_name / 'skeletons.npy'))  # (2, T, 34)
    fn_arr = np.load(str(SS_SKELETONS_GDINO / match_name / 'frame_nums.npy'))  # (T,)
    fn_set = {int(fn_arr[i]): i for i in range(len(fn_arr))}

    # Build per-rally hitter map (median-Y approach) — hitter is metadata only
    csv_dir = SS_CSV_ROOT / match_name
    hitter_map = ShuttleSetDataset._build_rally_hitter_map(csv_dir)

    # Parse CSV annotations
    for csv_path in sorted(csv_dir.glob('set*.csv')):
        try:
            set_num = int(''.join(filter(str.isdigit, csv_path.name))) or 0
        except ValueError:
            set_num = 0

        with open(csv_path) as f:
            for row in csv.DictReader(f):
                try:
                    rally      = int(row['rally'])
                    ball_round = int(float(row['ball_round']))
                    frame_num  = int(float(row['frame_num']))
                except (KeyError, ValueError, TypeError):
                    continue

                # Find nearest skeleton frame (stride-4 tolerance)
                t = fn_set.get(frame_num)
                if t is None:
                    candidates = fn_arr[np.abs(fn_arr - frame_num) <= 4]
                    if len(candidates) == 0:
                        continue
                    frame_num_matched = int(candidates[np.argmin(np.abs(candidates - frame_num))])
                    t = fn_set[frame_num_matched]

                # Slice shot_window centred on hit frame
                C, T_total, V = sk.shape
                half = shot_window // 2
                start = max(0, t - half)
                end = start + shot_window
                if end > T_total:
                    end = T_total
                    start = max(0, end - shot_window)
                segment = sk[:, start:end, :].copy()
                if segment.shape[1] < shot_window:
                    pad = np.zeros((C, shot_window - segment.shape[1], V), dtype=segment.dtype)
                    segment = np.concatenate([segment, pad], axis=1)

                # Hitter identity as metadata (top/bottom) — does NOT alter nodes
                player = row.get('player', '')
                rally_sides = hitter_map.get((set_num, rally), {})
                hitter = rally_sides.get(player, '')

                # Feature engineering
                features = feature_eng.compute(segment)
                x_tensor = torch.tensor(features, dtype=torch.float32)

                # Predict
                pred = predict_one(x_tensor, encoder, prototypes, shot_clf)

                # Ground truth shot type
                raw_type = row.get('type', '')
                unified  = SS_SHOT_TYPE_TO_UNIFIED.get(raw_type)

                pred.update({
                    'match':       match_name,
                    'set':         set_num,
                    'rally':       rally,
                    'ball_round':  ball_round,
                    'frame_num':   frame_num,
                    'player':      row.get('player', ''),
                    'hitter':      hitter,
                    'true_shot_type_zh':  raw_type,
                    'true_shot_type':     unified,
                    'shot_correct': (pred['pred_shot_type'] == unified) if (unified and pred['pred_shot_type']) else None,
                })
                ss_results.append(pred)

print(f"\nTotal SS held-out predictions: {len(ss_results)}")

In [ ]:
# Build DataFrame and save CSV
ss_df = pd.DataFrame(ss_results)

# Reorder columns
col_order = [
    'match', 'set', 'rally', 'ball_round', 'frame_num', 'player',
    'pred_strategy', 'strategy_conf', 'strategy_margin', 'strategy_2nd', 'strategy_2nd_conf',
    'pred_shot_type', 'shot_type_conf', 'true_shot_type', 'true_shot_type_zh', 'shot_correct',
]
ss_df = ss_df[[c for c in col_order if c in ss_df.columns]]

csv_path = RESULTS_DIR / f'heldout_ss_predictions_{FEATURE_LAYER}.csv'
ss_df.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}")
print(f"Shape: {ss_df.shape}")
ss_df.head(10)

### A.1 Shot-Type Accuracy

In [ ]:
from sklearn.metrics import f1_score, classification_report, confusion_matrix

if shot_clf is not None and len(ss_df) > 0:
    # Filter to shots with valid ground truth
    valid = ss_df[ss_df['true_shot_type'].notna() & ss_df['pred_shot_type'].notna()].copy()
    print(f"Shots with ground truth: {len(valid)}/{len(ss_df)}")

    if len(valid) > 0:
        y_true = valid['true_shot_type'].values
        y_pred = valid['pred_shot_type'].values

        accuracy = (y_true == y_pred).mean()
        all_types = sorted(set(y_true) | set(y_pred))
        macro_f1 = f1_score(y_true, y_pred, average='macro', labels=all_types, zero_division=0)
        weighted_f1 = f1_score(y_true, y_pred, average='weighted', labels=all_types, zero_division=0)

        print(f"\nShot-Type Metrics (SS Held-Out):")
        print(f"  Accuracy:    {accuracy:.3f}")
        print(f"  Macro-F1:    {macro_f1:.3f}")
        print(f"  Weighted-F1: {weighted_f1:.3f}")
        print(f"\n{classification_report(y_true, y_pred, labels=all_types, zero_division=0)}")

        # Confusion matrix
        cm = confusion_matrix(y_true, y_pred, labels=all_types)
        fig, ax = plt.subplots(figsize=(12, 10))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=all_types, yticklabels=all_types, ax=ax)
        ax.set_xlabel('Predicted Shot Type')
        ax.set_ylabel('True Shot Type')
        ax.set_title(f'SS Held-Out Shot-Type Confusion Matrix\n'
                     f'Accuracy: {accuracy:.3f} | Macro-F1: {macro_f1:.3f}')
        plt.xticks(rotation=45, ha='right')
        plt.yticks(rotation=0)
        plt.tight_layout()
        plt.savefig(RESULTS_DIR / f'heldout_ss_shot_type_confusion_{FEATURE_LAYER}.png', dpi=150)
        plt.show()
else:
    print("Shot-type evaluation skipped (no classifier or no data)")

### A.2 Strategy Distribution Analysis (ShuttleSet)

Since ShuttleSet has no strategy ground truth, we analyse the *predicted* distribution,
confidence levels, and cross-tabulation with shot type.

In [ ]:
if len(ss_df) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # A.2a — Strategy distribution per match (stacked bar)
    ax = axes[0, 0]
    ct = pd.crosstab(ss_df['match'].str[:30], ss_df['pred_strategy'], normalize='index')
    ct = ct.reindex(columns=STRATEGY_CLASSES, fill_value=0)
    ct.plot.barh(stacked=True, ax=ax, colormap='Set2')
    ax.set_xlabel('Proportion')
    ax.set_title('Predicted Strategy Distribution per Match')
    ax.legend(fontsize=8, loc='lower right')

    # A.2b — Strategy confidence histogram
    ax = axes[0, 1]
    for strat in STRATEGY_CLASSES:
        sub = ss_df[ss_df['pred_strategy'] == strat]['strategy_conf']
        if len(sub) > 0:
            ax.hist(sub, bins=20, alpha=0.5, label=strat, range=(0, 1))
    ax.set_xlabel('Confidence')
    ax.set_ylabel('Count')
    ax.set_title('Strategy Confidence Distribution')
    ax.legend(fontsize=8)
    ax.axvline(0.5, color='red', ls='--', alpha=0.6, label='margin=0.5')

    # A.2c — Strategy margin histogram
    ax = axes[1, 0]
    ax.hist(ss_df['strategy_margin'], bins=30, color='steelblue', edgecolor='white')
    ax.set_xlabel('Margin (top1 − top2 prob)')
    ax.set_ylabel('Count')
    ax.set_title('Strategy Prediction Margin')
    ax.axvline(cfg.proto.confidence_margin, color='red', ls='--', alpha=0.6)
    low_conf = (ss_df['strategy_margin'] < cfg.proto.confidence_margin).mean()
    ax.text(0.95, 0.95, f'{low_conf:.1%} below threshold',
            transform=ax.transAxes, ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round', fc='lightyellow'))

    # A.2d — Strategy × shot type cross-tabulation heatmap
    ax = axes[1, 1]
    valid_st = ss_df[ss_df['true_shot_type'].notna()]
    if len(valid_st) > 0:
        ct2 = pd.crosstab(valid_st['pred_strategy'], valid_st['true_shot_type'])
        # Normalize by column (shot type) to show: for each shot type, what strategy is predicted?
        ct2_norm = ct2.div(ct2.sum(axis=0), axis=1).fillna(0)
        sns.heatmap(ct2_norm, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax)
        ax.set_title('Strategy × Shot Type (col-normalised)')
        ax.set_xlabel('True Shot Type')
        ax.set_ylabel('Predicted Strategy')
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
    else:
        ax.text(0.5, 0.5, 'No valid shot types', ha='center', va='center', transform=ax.transAxes)

    plt.suptitle(f'ShuttleSet Held-Out — Strategy Analysis ({FEATURE_LAYER})', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f'heldout_ss_strategy_analysis_{FEATURE_LAYER}.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Summary stats
    print(f"\nStrategy distribution (overall):")
    print(ss_df['pred_strategy'].value_counts().to_string())
    print(f"\nMean confidence: {ss_df['strategy_conf'].mean():.3f}")
    print(f"Mean margin:     {ss_df['strategy_margin'].mean():.3f}")
    print(f"Low-confidence:  {low_conf:.1%} (margin < {cfg.proto.confidence_margin})")
else:
    print("No SS data to analyse")

---
## Part B — FineBadminton Held-Out Evaluation

Predict strategy on 10 held-out rallies (never seen during few-shot training).
Here we have **ground-truth strategy labels** so we compute full classification metrics.

In [ ]:
# Load FineBadminton dataset and get held-out split
fb_dataset = FineBadmintonDataset(
    feature_layer=FEATURE_LAYER,
    skeleton_source='gdino_v2',
    use_shuttle=cfg.ablation.use_shuttle,
)

train_idx, holdout_idx = fb_dataset.get_rally_splits(
    n_train_rallies=N_TRAIN_RALLIES, seed=FB_SPLIT_SEED
)

print(f"FB total shots:       {len(fb_dataset)}")
print(f"Train shot indices:   {len(train_idx)}")
print(f"Held-out shot indices: {len(holdout_idx)}")

# Show held-out rally IDs
holdout_rallies = sorted(set(fb_dataset.samples[i]['video'] for i in holdout_idx))
print(f"\nHeld-out rallies ({len(holdout_rallies)}):")
for r in holdout_rallies:
    n = sum(1 for i in holdout_idx if fb_dataset.samples[i]['video'] == r)
    print(f"  {r} ({n} shots)")

In [ ]:
# Run inference on FB held-out shots
fb_results = []

for idx in tqdm(holdout_idx, desc='FB held-out'):
    sample = fb_dataset.samples[idx]
    x_tensor, label = fb_dataset[idx]  # x_tensor: (C, T, V), label: int

    pred = predict_one(x_tensor, encoder, prototypes, shot_clf)

    pred.update({
        'rally':           sample['video'],
        'shot_idx':        sample.get('shot_idx', idx),
        'frame_start':     sample.get('frame_start', ''),
        'frame_end':       sample.get('frame_end', ''),
        'true_strategy':   IDX_TO_STRATEGY[label],
        'true_hit_type':   sample.get('hit_type', ''),
        'strategy_correct': pred['pred_strategy'] == IDX_TO_STRATEGY[label],
    })

    # Map FB shot type to unified — prefer subtype (from dataset.py), fallback to hit_type
    unified_gt = sample.get('unified_shot_type')  # already resolved by dataset.py (subtype-first)
    pred['true_shot_type'] = unified_gt
    pred['shot_correct'] = (pred['pred_shot_type'] == unified_gt) if (unified_gt and pred['pred_shot_type']) else None

    fb_results.append(pred)

print(f"\nTotal FB held-out predictions: {len(fb_results)}")
print(f"Strategy accuracy: {sum(r['strategy_correct'] for r in fb_results) / len(fb_results):.3f}")

In [ ]:
# Build DataFrame and save CSV
fb_df = pd.DataFrame(fb_results)

col_order = [
    'rally', 'shot_idx', 'frame_start', 'frame_end',
    'true_strategy', 'pred_strategy', 'strategy_correct', 'strategy_conf', 'strategy_margin',
    'strategy_2nd', 'strategy_2nd_conf',
    'true_hit_type', 'true_shot_type', 'pred_shot_type', 'shot_type_conf', 'shot_correct',
]
fb_df = fb_df[[c for c in col_order if c in fb_df.columns]]

csv_path = RESULTS_DIR / f'heldout_fb_predictions_{FEATURE_LAYER}.csv'
fb_df.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}")
fb_df.head(10)

### B.1 Strategy Classification Metrics (FineBadminton)

In [ ]:
if len(fb_df) > 0:
    y_true = fb_df['true_strategy'].values
    y_pred = fb_df['pred_strategy'].values

    accuracy = (y_true == y_pred).mean()
    macro_f1 = f1_score(y_true, y_pred, average='macro', labels=STRATEGY_CLASSES, zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average='weighted', labels=STRATEGY_CLASSES, zero_division=0)

    print(f"FB Held-Out Strategy Metrics:")
    print(f"  Accuracy:    {accuracy:.3f}")
    print(f"  Macro-F1:    {macro_f1:.3f}")
    print(f"  Weighted-F1: {weighted_f1:.3f}")
    print(f"\n{classification_report(y_true, y_pred, labels=STRATEGY_CLASSES, zero_division=0)}")

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=STRATEGY_CLASSES)
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Raw counts
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=STRATEGY_CLASSES, yticklabels=STRATEGY_CLASSES, ax=axes[0])
    axes[0].set_xlabel('Predicted Strategy')
    axes[0].set_ylabel('True Strategy')
    axes[0].set_title(f'Strategy Confusion Matrix (counts)\nAccuracy: {accuracy:.3f}')

    # Row-normalised
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    cm_norm = np.nan_to_num(cm_norm)
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=STRATEGY_CLASSES, yticklabels=STRATEGY_CLASSES, ax=axes[1])
    axes[1].set_xlabel('Predicted Strategy')
    axes[1].set_ylabel('True Strategy')
    axes[1].set_title(f'Strategy Confusion Matrix (row-normalised)\nMacro-F1: {macro_f1:.3f}')

    for ax in axes:
        plt.setp(ax.get_xticklabels(), rotation=30, ha='right')

    plt.suptitle(f'FineBadminton Held-Out — {FEATURE_LAYER}', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f'heldout_fb_strategy_confusion_{FEATURE_LAYER}.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Per-rally breakdown
    print("\nPer-rally accuracy:")
    for rally in holdout_rallies:
        sub = fb_df[fb_df['rally'] == rally]
        racc = (sub['true_strategy'] == sub['pred_strategy']).mean()
        print(f"  {rally}: {racc:.3f} ({len(sub)} shots)")
else:
    print("No FB data")

### B.2 Confidence Analysis (FineBadminton)

Compare confidence/margin distributions for correct vs incorrect predictions.

In [ ]:
if len(fb_df) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # B.2a — Confidence: correct vs incorrect
    ax = axes[0]
    correct = fb_df[fb_df['strategy_correct']]['strategy_conf']
    wrong   = fb_df[~fb_df['strategy_correct']]['strategy_conf']
    ax.hist(correct, bins=15, alpha=0.6, label=f'Correct (n={len(correct)})', color='green', range=(0,1))
    ax.hist(wrong,   bins=15, alpha=0.6, label=f'Wrong (n={len(wrong)})',     color='red',   range=(0,1))
    ax.set_xlabel('Confidence')
    ax.set_ylabel('Count')
    ax.set_title('Confidence: Correct vs Wrong')
    ax.legend()

    # B.2b — Margin: correct vs incorrect
    ax = axes[1]
    ax.hist(correct.index.map(lambda i: fb_df.loc[i, 'strategy_margin']),
            bins=15, alpha=0.6, label='Correct', color='green', range=(0,1))
    ax.hist(wrong.index.map(lambda i: fb_df.loc[i, 'strategy_margin']),
            bins=15, alpha=0.6, label='Wrong', color='red', range=(0,1))
    ax.set_xlabel('Margin (top1 - top2)')
    ax.set_ylabel('Count')
    ax.set_title('Margin: Correct vs Wrong')
    ax.legend()
    ax.axvline(cfg.proto.confidence_margin, color='black', ls='--', alpha=0.5)

    # B.2c — Accuracy vs confidence threshold
    ax = axes[2]
    thresholds = np.arange(0.2, 0.95, 0.05)
    accs, coverages = [], []
    for th in thresholds:
        mask = fb_df['strategy_conf'] >= th
        if mask.sum() > 0:
            accs.append(fb_df.loc[mask, 'strategy_correct'].mean())
            coverages.append(mask.mean())
        else:
            accs.append(np.nan)
            coverages.append(0)
    ax.plot(thresholds, accs, 'b-o', label='Accuracy', markersize=4)
    ax2 = ax.twinx()
    ax2.plot(thresholds, coverages, 'r--s', label='Coverage', markersize=4)
    ax.set_xlabel('Confidence Threshold')
    ax.set_ylabel('Accuracy', color='blue')
    ax2.set_ylabel('Coverage', color='red')
    ax.set_title('Accuracy vs Coverage (confidence filtering)')
    ax.legend(loc='lower left')
    ax2.legend(loc='lower right')

    plt.suptitle(f'FineBadminton Held-Out — Confidence Analysis ({FEATURE_LAYER})', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f'heldout_fb_confidence_{FEATURE_LAYER}.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## Part C — Combined Summary + Expert Review Export

Merge SS and FB predictions into a single CSV sorted by confidence (lowest first)
for expert review of the most uncertain predictions.

In [ ]:
# Combine both datasets for expert review
dfs = []

if len(ss_df) > 0:
    ss_export = ss_df.copy()
    ss_export['dataset'] = 'ShuttleSet'
    ss_export['sample_id'] = ss_export.apply(
        lambda r: f"SS|{r['match'][:30]}|s{r['set']}r{r['rally']}|b{r['ball_round']}", axis=1
    )
    ss_export['true_strategy'] = None  # no ground truth for SS
    ss_export['strategy_correct'] = None
    dfs.append(ss_export)

if len(fb_df) > 0:
    fb_export = fb_df.copy()
    fb_export['dataset'] = 'FineBadminton'
    fb_export['sample_id'] = fb_export.apply(
        lambda r: f"FB|{r['rally']}|shot{r['shot_idx']}", axis=1
    )
    fb_export['match'] = fb_export['rally']
    dfs.append(fb_export)

if dfs:
    combined = pd.concat(dfs, ignore_index=True)

    # Select and order columns for export
    export_cols = [
        'dataset', 'sample_id', 'match',
        'pred_strategy', 'strategy_conf', 'strategy_margin',
        'strategy_2nd', 'strategy_2nd_conf',
        'true_strategy', 'strategy_correct',
        'pred_shot_type', 'shot_type_conf', 'true_shot_type', 'shot_correct',
    ]
    export_df = combined[[c for c in export_cols if c in combined.columns]].copy()

    # Sort by confidence ascending (most uncertain first for expert review)
    export_df = export_df.sort_values('strategy_conf', ascending=True).reset_index(drop=True)

    # Add expert review columns (blank for human to fill in)
    export_df['expert_strategy'] = ''
    export_df['expert_notes'] = ''

    csv_path = RESULTS_DIR / f'heldout_expert_review_{FEATURE_LAYER}.csv'
    export_df.to_csv(csv_path, index=False)
    print(f"Saved: {csv_path}")
    print(f"Total samples: {len(export_df)}")
    print(f"\nTop 10 lowest-confidence predictions (for expert review):")
    display(export_df.head(10))
else:
    print("No data to export")

In [ ]:
# Final summary dashboard
print("=" * 70)
print(f"  HELD-OUT EVALUATION SUMMARY  ({FEATURE_LAYER})")
print("=" * 70)

if len(ss_df) > 0:
    valid_shots = ss_df[ss_df['shot_correct'].notna()]
    shot_acc = valid_shots['shot_correct'].mean() if len(valid_shots) > 0 else float('nan')
    print(f"\n  ShuttleSet ({len(available_matches)} matches, {len(ss_df)} shots):")
    print(f"    Shot-type accuracy:          {shot_acc:.3f}")
    print(f"    Strategy mean confidence:    {ss_df['strategy_conf'].mean():.3f}")
    print(f"    Strategy mean margin:        {ss_df['strategy_margin'].mean():.3f}")
    print(f"    Low-confidence shots:        {(ss_df['strategy_margin'] < cfg.proto.confidence_margin).sum()}"
          f" ({(ss_df['strategy_margin'] < cfg.proto.confidence_margin).mean():.1%})")

if len(fb_df) > 0:
    fb_acc = fb_df['strategy_correct'].mean()
    fb_macro = f1_score(fb_df['true_strategy'], fb_df['pred_strategy'],
                        average='macro', labels=STRATEGY_CLASSES, zero_division=0)
    print(f"\n  FineBadminton ({len(holdout_rallies)} rallies, {len(fb_df)} shots):")
    print(f"    Strategy accuracy:           {fb_acc:.3f}")
    print(f"    Strategy macro-F1:           {fb_macro:.3f}")
    print(f"    Strategy mean confidence:    {fb_df['strategy_conf'].mean():.3f}")
    print(f"    Strategy mean margin:        {fb_df['strategy_margin'].mean():.3f}")
    fb_valid_shots = fb_df[fb_df['shot_correct'].notna()]
    if len(fb_valid_shots) > 0:
        print(f"    Shot-type accuracy:          {fb_valid_shots['shot_correct'].mean():.3f}")

print(f"\n  Outputs saved to: {RESULTS_DIR}/")
print(f"    heldout_ss_predictions_{FEATURE_LAYER}.csv")
print(f"    heldout_fb_predictions_{FEATURE_LAYER}.csv")
print(f"    heldout_expert_review_{FEATURE_LAYER}.csv  (sorted by confidence, lowest first)")
print("=" * 70)